# PPXFL Colab experiment runner

Mounts Drive (for persistent data/results across sessions that can die anytime), pulls the repo, installs deps, and runs whatever experiment cells aren't done yet via `run_experiments.py`. Safe to re-run this whole notebook after a session dies — every training script checkpoints internally and the driver skips completed cells.

**One-time setup before first run:** create a folder `PPXFL_runs/` in your Drive root, and inside it clone (or upload) this repo as `PPXFL_runs/ppxfl-alzheimer/`, OR just let the git-clone cell below do it (edit the URL first).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/PPXFL_runs'
REPO_DIR = f'{DRIVE_ROOT}/ppxfl-alzheimer'

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive root:', DRIVE_ROOT)

In [ ]:
# First-time only: clone the repo into Drive. If it already exists, pull latest instead.
# EDIT THIS URL to your actual GitHub remote before running.
GIT_REMOTE = 'https://github.com/<your-username>/ppxfl-alzheimer.git'

if not os.path.isdir(REPO_DIR):
    !git clone {GIT_REMOTE} "{REPO_DIR}"
else:
    !cd "{REPO_DIR}" && git pull

In [ ]:
# Install dependencies (Colab already ships a CUDA-enabled torch — do NOT reinstall
# torch/torchvision here, that's exactly what broke the local venv; just add the rest).
!pip install -q flwr opacus captum shap nibabel nilearn SimpleITK scikit-learn pandas \
    matplotlib seaborn grad-cam scikit-image tqdm

import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# Data: if data/processed, data/splits, data/clients aren't already in the Drive repo copy
# (they're large — commit them to Drive once, not to git), copy them in now.
# Skip this cell if the repo in Drive already has data/ populated from a previous session.
import os
data_dir = f'{REPO_DIR}/data/processed'
print('manifest present:', os.path.exists(f'{data_dir}/manifest.csv'))
print('splits present:', os.path.exists(f'{REPO_DIR}/data/splits/splits_v1.json'))
# If either is missing, upload data/ from local (or re-run preprocess.py + splits.py here
# against NIfTI files you've also placed in Drive — same commands as local, see PROGRESS.md).

In [ ]:
# Show what's pending, then run it. Each cell in the experiment matrix checkpoints
# internally (epoch/round-level, atomic writes) so a Colab disconnect mid-cell loses
# at most one epoch/round of progress, not the whole cell.
!cd "{REPO_DIR}" && python run_experiments.py --list

In [ ]:
# Run everything pending. If the session is about to time out, just stop this cell —
# re-running the notebook from the top later resumes exactly where it left off.
!cd "{REPO_DIR}" && python run_experiments.py

In [ ]:
# Optional: run only a specific matrix group, e.g. just the DP-FL sweep
# !cd "{REPO_DIR}" && python run_experiments.py --only B7